In [ ]:
%%capture
!pip install parler-tts
!pip install "protobuf>=5.28.0" --upgrade
# !pip install parler-tts transformers

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# hf_token = userdata.get('HF_TOKEN')
# login(token=hf_token)

from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')\
login(token=hf_token)

# !pip install git+https://huggingface.co/huggingface/parler-tts.git

In [ ]:
from google.colab import userdata
from huggingface_hub import login
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer
import torch, numpy as np, soundfile as sf
import IPython.display as ipd

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

# MODEL_ID      = "milanakdj/indic-parler-tts-nepali-finetuned-v55-v1"
# MODEL_ID = "milanakdj/nepali-parler-tts-epoc-2.5-checkpoint"
MODEL_ID = "milanakdj/indic-parler-tts-nepali-finetuned-dgx-v2-test"
# MODEL_ID      = "ai4bharat/indic-parler-tts"
BASE_MODEL_ID = "ai4bharat/indic-parler-tts-pretrained"

device = "cuda" if torch.cuda.is_available() else "cpu"

infer_model = ParlerTTSForConditionalGeneration.from_pretrained(
    MODEL_ID, token=hf_token
).to(device).eval()

# ✅ Prompt tokenizer: must come from the base model (Indic tokenizer)
#    The HF repo root has flan-t5-large (overwritten by desc tokenizer push)
infer_prompt_tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=hf_token)

# ✅ Description tokenizer: flan-t5-large, which IS what's at the HF repo root
infer_desc_tok = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)

TEST_TEXTS = [
    "देवनागरी लिपि अथवा नागरी लिपि बायाँ देखि दायाँ सम्म लेखिने अक्षरात्मक लिपिमा आधारित एक प्राचीन ब्राह्मी लिपि हो। यसको प्रयोग भारतीय उपमहाद्वीपमा हुने गर्दछ। यसको विकास प्राचीन भारतमा पहिलो देखि चौथो शताब्दीको बीचमा भएको थियो भने यसको प्रयोग सातौँ शताब्दी देखी हुँदै आएको छ। देवनागरी लिपिमा, ४७ वटा वर्णहरू हुन्छन् जसमा १४ वटा स्वरवर्ण र ३३ वटा व्यञ्जनवर्ण हुन्छन्। यो संसारमा सबैभन्दा बढी प्रयोग हुने लिपि मध्ये चौथो हो जहा यस लिपिमा १२० भन्दा बढी भाषाहरू लेखिन्छ। यो लिपि नेपाल र भारत गणतन्त्रको आधिकारिक लिपिहरू मध्ये एक हो।",
    "काठमाडौं नेपालको राजधानी र सबैभन्दा ठूलो सहर हो।",
    "मलाई नेपाली भाषा मन पर्छ।",
]
TEST_DESC = "Shristi speaks with a deep, formal Nepali voice. Her speech is clear, steady and authoritative with natural pacing in a quiet noise-free environment."

for i, text in enumerate(TEST_TEXTS):
    print(f"\n[{i+1}] {text}")
    desc_ids   = infer_desc_tok(TEST_DESC, return_tensors="pt").input_ids.to(device)
    prompt_ids = infer_prompt_tok(text, return_tensors="pt").input_ids.to(device)

    with torch.inference_mode():
        gen = infer_model.generate(
            input_ids=desc_ids,
            prompt_input_ids=prompt_ids,
            do_sample=True,
            temperature=1.0,
            max_new_tokens=500,
        )

    audio = gen.cpu().numpy().squeeze().astype(np.float32)
    max_val = np.abs(audio).max()
    if max_val > 1e-6:
        audio = audio / max_val

    fname = f"test_nepali_{i+1}.wav"
    sf.write(fname, audio, infer_model.config.sampling_rate)
    print(f"  💾 Saved: {fname}")
    ipd.display(ipd.Audio(audio, rate=infer_model.config.sampling_rate))

print("\n✅ Inference complete!")

In [ ]:
TRAINING_DESCRIPTIONS = [
    "Shristi speaks with a deep, formal Nepali voice. Her speech is clear, steady and authoritative with natural pacing in a quiet noise-free environment."
    "A female speaker delivers clear Nepali speech at a normal pace. The recording is of very high quality.",
    "A female speaker speaks clearly and naturally in Nepali. The audio is clean with no background noise.",
    "A neutral speaker reads Nepali text at a steady pace. The recording quality is excellent.",
    "A female speaker delivers speech slowly and clearly in Nepali. Very high quality audio.",
]

text = "सबैलाई नमस्कार, मैले गएको वर्ष हिमालयमा एउटा येति देखेको थिएँ। त्यो साँच्चिकै प्राणी हो। "

for j, desc in enumerate(TRAINING_DESCRIPTIONS):
    desc_ids   = infer_desc_tok(desc, return_tensors="pt").input_ids.to(device)
    prompt_ids = infer_prompt_tok(text, return_tensors="pt").input_ids.to(device)

    with torch.inference_mode():
        gen = infer_model.generate(
            input_ids=desc_ids,
            prompt_input_ids=prompt_ids,
            do_sample=True,
            temperature=1,
            max_new_tokens=1000,
        )

    audio = gen.cpu().numpy().squeeze().astype(np.float32)
    max_val = np.abs(audio).max()
    if max_val > 1e-6:
        audio = audio / max_val

    print(f"\n[Desc {j+1}] {desc}")
    ipd.display(ipd.Audio(audio, rate=infer_model.config.sampling_rate))


In [ ]:
TRAINING_DESCRIPTIONS = [
    "Shristi speaks with a deep, formal Nepali voice. Her speech is clear, steady and authoritative with natural pacing in a quiet noise-free environment.",

    # "A female speaker speaks clearly and naturally in Nepali. The audio is clean with no background noise.",

    # "A neutral speaker reads Nepali text at a steady pace. The recording quality is excellent.",

    # "A female speaker delivers speech slowly and clearly in Nepali. Very high quality audio.",

    # # New scared/emotional prompt
    # "A frightened female speaker tells a shocking story in Nepali with fear and disbelief in her voice. She emphasizes the word 'Yeti' strongly. The speech sounds emotional, tense, and realistic. Very high quality clean audio.",
]

text = "तिमीहरूलाई विश्वास नलाग्ला... तर मैले हिमालयमा आफ्नै आँखाले येति देखेको थिएँ। त्यो... साँच्चिकै थियो!"

sentences = [
    "नमस्ते, तपाईंलाई आज कस्तो सहयोग चाहिन्छ?",
    "म तपाईंको सहायक बोल्दैछु।",
    "कृपया आफ्नो समस्या विस्तारमा बताइदिनुहोस्।",
    "आजको मौसम निकै राम्रो देखिन्छ।",
    "तपाईंको दिन शुभ रहोस्।",
    "म नेपाली भाषामा पनि कुरा गर्न सक्छु।",
    "के तपाईंलाई कुनै जानकारी चाहिएको छ?",
    "तपाईंले पठाएको अनुरोध प्रक्रिया हुँदैछ।",
    "कृपया केही क्षण प्रतीक्षा गर्नुहोस्।",
    "धन्यवाद, तपाईंको सन्देश प्राप्त भयो।",
    "यो एउटा परीक्षण वाक्य हो।",
    "कम्प्युटर विज्ञान निकै रोचक विषय हो।",
    "म नयाँ प्रविधिहरू सिक्दैछु।",
    "नेपाल प्राकृतिक सौन्दर्यले भरिएको देश हो।",
    "काठमाडौं नेपालको राजधानी शहर हो।",
    "आज तपाईंले के सिक्नुभयो?",
    "संगीत सुन्न मलाई मन पर्छ।",
    "कृत्रिम बुद्धिमत्ता भविष्यको महत्वपूर्ण प्रविधि हो।",
    "तपाईंको इन्टरनेट जडान स्थिर देखिन्छ।",
    "कृपया फेरि प्रयास गर्नुहोस्।",
    "यो आवाज परीक्षणको लागि प्रयोग गरिएको वाक्य हो।",
    "विद्यालयमा विद्यार्थीहरू अध्ययन गर्दैछन्।",
    "हामी नयाँ परियोजनामा काम गरिरहेका छौं।",
    "तपाईंको फाइल सफलतापूर्वक अपलोड भयो।",
    "अब म अर्को वाक्य पढ्दैछु।",
    "तपाईंलाई सहयोग गर्न पाउँदा खुशी लाग्यो।",
    "सुरक्षित यात्रा गर्नुहोस्।",
    "तपाईंको अर्डर तयार हुँदैछ।",
    "कृपया आफ्नो नाम भन्नुहोस्।",
    "यो प्रणाली अहिले सक्रिय अवस्थामा छ।"
]

desc_ids   = infer_desc_tok(desc, return_tensors="pt").input_ids.to(device)
for j, desc in enumerate(sentences):
    prompt_ids = infer_prompt_tok(desc, return_tensors="pt").input_ids.to(device)

    with torch.inference_mode():
        gen = infer_model.generate(
            input_ids=desc_ids,
            prompt_input_ids=prompt_ids,
            do_sample=True,
            temperature=1,
            max_new_tokens=2000,
        )

    audio = gen.cpu().numpy().squeeze().astype(np.float32)
    max_val = np.abs(audio).max()
    if max_val > 1e-6:
        audio = audio / max_val

    print(f"\n[Desc {j+1}] {desc}")
    ipd.display(ipd.Audio(audio, rate=infer_model.config.sampling_rate))